# 01. Setup & Baseline (Image Cell)

**Phase 1**: torchvision 데이터셋 로드 → DSC 베이스라인 → 모델 5개 베이스라인

DSC v5 framework — image × classification cell (ADR-014 사전등록).

---

## 0. 환경 설정

In [4]:
# ============================================================
# 0-1. Drive 마운트 + GPU 확인
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import numpy as np
import torch

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
print(f'torch: {torch.__version__}')

Mounted at /content/drive
device: cpu
torch: 2.10.0+cpu


In [5]:
# ============================================================
# 0-2. 의존성 설치 (Colab 환경)
# ============================================================
%pip install -q timm imagehash opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 5.8 MB/s eta 0:00:00


## 1. 데이터셋 로드 — CIFAR-10 / Fashion-MNIST / Flowers102

In [6]:
# ============================================================
# 1-1. 데이터셋 사전등록 메타
# ============================================================
DATASETS = {
    'CIFAR10': {
        'loader': 'torchvision.datasets.CIFAR10',
        'n_classes': 10, 'image_size': 32, 'channels': 3,
    },
    'FashionMNIST': {
        'loader': 'torchvision.datasets.FashionMNIST',
        'n_classes': 10, 'image_size': 28, 'channels': 1,
    },
    'Flowers102': {
        'loader': 'torchvision.datasets.Flowers102',
        'n_classes': 102, 'image_size': 224, 'channels': 3,
    },
}
print(f'데이터셋: {list(DATASETS.keys())}')

데이터셋: ['CIFAR10', 'FashionMNIST', 'Flowers102']


In [7]:
# ============================================================
# 1-2. 데이터셋 다운로드 (torchvision)
# ============================================================
import torchvision

def load_dataset(ds_name, train=True):
    if ds_name == 'CIFAR10':
        return torchvision.datasets.CIFAR10(f'{DATA_DIR}/CIFAR10', train=train, download=True)
    elif ds_name == 'FashionMNIST':
        return torchvision.datasets.FashionMNIST(f'{DATA_DIR}/FashionMNIST', train=train, download=True)
    elif ds_name == 'Flowers102':
        split = 'train' if train else 'test'
        return torchvision.datasets.Flowers102(f'{DATA_DIR}/Flowers102', split=split, download=True)
    raise ValueError(ds_name)

datasets_loaded = {}
for ds_name in DATASETS:
    print(f'\n{ds_name} 로드...')
    train_ds = load_dataset(ds_name, train=True)
    test_ds = load_dataset(ds_name, train=False)
    datasets_loaded[ds_name] = (train_ds, test_ds)
    print(f'  train: {len(train_ds)}, test: {len(test_ds)}')


CIFAR10 로드...
  train: 50000, test: 10000

FashionMNIST 로드...
  train: 60000, test: 10000

Flowers102 로드...
  train: 1020, test: 6149


In [8]:
# ============================================================
# 1-3. PIL → numpy 추출 helper
# ============================================================
def dataset_to_arrays(ds, sample_cap=None, random_state=1):
    images, labels = [], []
    n = len(ds) if sample_cap is None else min(len(ds), sample_cap)
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(ds))[:n] if sample_cap else range(n)
    for i in idx:
        img, lbl = ds[i]
        images.append(np.array(img))
        labels.append(int(lbl))
    return images, labels

# 각 데이터셋의 train sample (DSC 계산용 — 큰 데이터는 sample_cap 적용)
SAMPLE_CAP = 5000  # DSC 메트릭 sample_cap (메모리/시간 절약)
print(f'DSC sample_cap = {SAMPLE_CAP} (이미지 수)')

DSC sample_cap = 5000 (이미지 수)


## 2. DSC 베이스라인 (clean train data)

In [9]:
# ============================================================
# 2-1. DSC framework import + 베이스라인 계산
# ============================================================
from dsc_framework import compute_dsc_image, DEFAULT_WEIGHTS_IMAGE

print('이미지 cell DSC 엔진 import 완료')
print(f'사전등록 가중치 (sum={sum(DEFAULT_WEIGHTS_IMAGE.values()):.2f}):')
for k, v in DEFAULT_WEIGHTS_IMAGE.items():
    print(f'  {k:<35s} {v:.2f}')

이미지 cell DSC 엔진 import 완료
사전등록 가중치 (sum=1.00):
  completeness_image                  0.15
  uniqueness                          0.10
  validity                            0.05
  consistency                         0.05
  outlier_ratio                       0.05
  class_balance                       0.10
  feature_correlation                 0.05
  label_consistency                   0.20
  feature_informativeness             0.10
  sample_quality_image                0.15


In [10]:
# ============================================================
# 2-2. 데이터셋별 베이스라인 DSC
# ============================================================
baseline_dsc_rows = []
for ds_name, (train_ds, _) in datasets_loaded.items():
    print(f'\n{ds_name} DSC 계산 중...')
    images, labels = dataset_to_arrays(train_ds, sample_cap=SAMPLE_CAP)
    res = compute_dsc_image(images, labels, sample_cap=SAMPLE_CAP)
    print(f'  DSC = {res["score"]} ({res["grade"]})')
    baseline_dsc_rows.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0, **res})

import pandas as pd
df_baseline_dsc = pd.DataFrame(baseline_dsc_rows)
df_baseline_dsc


CIFAR10 DSC 계산 중...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 65.9MB/s]


  DSC = 91.35 (A)

FashionMNIST DSC 계산 중...
  DSC = 86.69 (B)

Flowers102 DSC 계산 중...
  DSC = 80.88 (B)


,dataset,polluter,level,score,grade,completeness_image,uniqueness,validity,consistency,outlier_ratio,class_balance,sample_quality_image,feature_correlation,label_consistency,feature_informativeness
0,CIFAR10,none,0.0,91.35,A,0.9966,1.000,1.0,1.0000,0.9870,0.948,0.9357,1.0,0.6476,1.0
1,FashionMNIST,none,0.0,86.69,B,0.4836,0.999,1.0,1.0000,0.9996,0.930,0.9940,1.0,0.7617,1.0
2,Flowers102,none,0.0,80.88,B,0.9843,0.999,1.0,0.1872,0.9873,1.000,0.7928,1.0,0.4178,1.0


## 3. 베이스라인 모델 학습 (5 모델 × 3 데이터셋)

GPU 시간 절약: epochs=10 sanity check, 정식 평가는 03 노트북에서 epochs=30.

In [11]:
# ============================================================
# 3-1. 모델 정의 (사전등록)
# ============================================================
import torch.nn as nn
import torchvision.models as tvm

def get_model(model_name, n_classes, in_channels=3, image_size=32):
    if model_name == 'ResNet18':
        m = tvm.resnet18(weights=None)
        if in_channels != 3:
            m.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        return m
    if model_name == 'EfficientNetB0':
        m = tvm.efficientnet_b0(weights=None)
        if in_channels != 3:
            m.features[0][0] = nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1, bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        return m
    if model_name == 'MobileNetV3small':
        m = tvm.mobilenet_v3_small(weights=None)
        if in_channels != 3:
            m.features[0][0] = nn.Conv2d(in_channels, 16, kernel_size=3, stride=2, padding=1, bias=False)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, n_classes)
        return m
    if model_name == 'ViTTiny':
        import timm
        return timm.create_model('vit_tiny_patch16_224', pretrained=False,
                                 num_classes=n_classes, in_chans=in_channels)
    if model_name == 'CNNSimple':
        return nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes),
        )
    raise ValueError(model_name)

MODEL_NAMES = ['ResNet18', 'EfficientNetB0', 'MobileNetV3small', 'ViTTiny', 'CNNSimple']
print(f'모델 5개: {MODEL_NAMES}')

모델 5개: ['ResNet18', 'EfficientNetB0', 'MobileNetV3small', 'ViTTiny', 'CNNSimple']


In [ ]:
# ============================================================
# 3-2. 학습 루프 (epochs=5, train_cap=5000, test_cap=2000)
# ============================================================
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

def get_transform(ds_name, train=True):
    meta = DATASETS[ds_name]
    size = max(meta['image_size'], 224 if 'ViT' in '|'.join(MODEL_NAMES) else meta['image_size'])
    tfs = [T.Resize((size, size)), T.ToTensor()]
    if meta['channels'] == 1:
        tfs.append(T.Lambda(lambda x: x.repeat(3, 1, 1)))
    tfs.append(T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    return T.Compose(tfs)


class WrappedDataset(torch.utils.data.Dataset):
    def __init__(self, base, transform):
        self.base = base; self.transform = transform
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, lbl = self.base[i]
        if self.transform: img = self.transform(img)
        return img, lbl


def train_eval(model, train_ds, test_ds, epochs=5, batch_size=128, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=2)
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = crit(logits, y)
            loss.backward()
            opt.step()
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


from time import time
EPOCHS_BASELINE = 5
TRAIN_SAMPLE_CAP = 5000
TEST_SAMPLE_CAP = 2000

rng_split = np.random.RandomState(42)
baseline_perf_rows = []
for ds_name, (train_raw, test_raw) in datasets_loaded.items():
    transform = get_transform(ds_name)

    train_idx = rng_split.permutation(len(train_raw))[:min(TRAIN_SAMPLE_CAP, len(train_raw))]
    test_idx  = rng_split.permutation(len(test_raw))[:min(TEST_SAMPLE_CAP, len(test_raw))]
    train_ds = WrappedDataset(Subset(train_raw, train_idx), transform)
    test_ds  = WrappedDataset(Subset(test_raw,  test_idx),  transform)
    meta = DATASETS[ds_name]

    for model_name in MODEL_NAMES:
        t0 = time()
        try:
            model = get_model(model_name, meta['n_classes'], in_channels=3,
                              image_size=meta['image_size'])
            acc = train_eval(model, train_ds, test_ds, epochs=EPOCHS_BASELINE)
        except Exception as e:
            print(f'  [{ds_name}/{model_name}] 학습 실패: {e}')
            acc = float('nan')
        elapsed = time() - t0
        baseline_perf_rows.append({
            'dataset': ds_name, 'polluter': 'none', 'level': 0.0,
            'model': model_name,
            'accuracy': round(acc, 4) if acc == acc else None,
            'epochs': EPOCHS_BASELINE,
            'train_n': len(train_ds), 'test_n': len(test_ds),
        })
        print(f'  [{ds_name}/{model_name:<18s}] acc={acc:.4f}  ({elapsed:.0f}s)')

df_baseline_perf = pd.DataFrame(baseline_perf_rows)
df_baseline_perf


## 4. 결과 저장

In [ ]:
# ============================================================
# 4-1. 결과 저장 (이미지 cell — 별도 파일)
# ============================================================
dsc_path = f'{RESULTS_DIR}/dsc_scores_image.csv'
perf_path = f'{RESULTS_DIR}/model_performance_image.csv'

def upsert_baseline(path, new_df):
    if os.path.isfile(path):
        existing = pd.read_csv(path)
        baseline_mask = (existing.polluter == 'none') & (existing.level == 0.0)
        kept = existing[~baseline_mask].copy()
        for col in new_df.columns:
            if col not in kept.columns:
                kept[col] = pd.NA
        extra = [c for c in kept.columns if c not in new_df.columns]
        kept = kept[list(new_df.columns) + extra]
        combined = pd.concat([kept, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(path, index=False)
    return len(combined)

n1 = upsert_baseline(dsc_path, df_baseline_dsc)
n2 = upsert_baseline(perf_path, df_baseline_perf)
print(f'DSC 저장: {dsc_path} (총 {n1}건)')
print(f'모델 성능 저장: {perf_path} (총 {n2}건)')
print('--- 노트북 01 이미지 cell 완료 ---')